# Notebook 22 — Combining datasets across platforms

PSF pathway scores computed on 10x Chromium, Smart-seq2, bulk RNA-seq,
and spatial transcriptomics are *directionally* comparable after standard
preprocessing, but per-cell magnitudes drift with platform — gene dropout
patterns and technology-specific biases couple with cell biology in ways
that plain z-scoring cannot remove.

This notebook walks through PSF's v0.6 cross-platform harmonization layer
(`pathway_subtyping.harmonize`), which combines a platform-invariant cell
embedding (UCE in production, `FallbackEmbedder` for local testing) with
a per-platform linear aligner to put all platforms in a common frame.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathway_subtyping.harmonize import (
    CrossPlatformAligner,
    CrossPlatformBenchmark,
    FallbackEmbedder,
    HarmonizationReport,
    simulate_platform_distortion,
)

rng = np.random.default_rng(42)

## 1. Synthetic reference pathway-score matrix

A tiny synthetic cohort with three biological clusters and 20 pathways.
Stands in for a real pathway-score matrix you would produce via PSF's
`score_pathways_from_expression` on any real cohort.

In [ ]:
n_cells, n_pathways = 400, 20
cluster_means = rng.standard_normal((3, n_pathways)) * 1.5
cluster_assignments = rng.integers(0, 3, size=n_cells)
base = cluster_means[cluster_assignments]
noise = rng.standard_normal((n_cells, n_pathways)) * 0.3
reference_scores = pd.DataFrame(
    base + noise,
    columns=[f'PATH_{i}' for i in range(n_pathways)],
)
reference_scores.head()

## 2. Simulate platform-specific distortion

The distortion has three components: a fixed per-pathway shift (platform
detection bias), an embedding-dependent per-cell shift (platform bias
interacting with cell biology), and i.i.d. noise. `CrossPlatformBenchmark`
bundles this so you can pass any reference matrix and platform list.

In [ ]:
bench = CrossPlatformBenchmark(
    reference_scores=reference_scores,
    platforms=['10x', 'smartseq2', 'bulk_rnaseq', 'spatial'],
)
result = bench.run(seed=0)
print(f'pre  rho across platform pairs: {result["pre_rho"]:.3f}')
print(f'post rho across platform pairs: {result["post_rho"]:.3f}')
print(f'improvement: {result["improvement"]:+.3f}')
print()
print('per-pair Spearman rho (pre -> post):')
for pair, stats in result['pair_details'].items():
    print(f'  {pair:30s} {stats["pre"]:+.3f} -> {stats["post"]:+.3f}')

## 3. Inspect per-platform drift

`HarmonizationReport` summarises how much each pathway score had to move
per platform to align with the reference. Use this to identify which
pathways are most platform-sensitive.

In [ ]:
report: HarmonizationReport = result['report']
print(report.summary())
print()
print('mean absolute drift per platform:')
for plat, mean_drift in report.mean_platform_drift.items():
    print(f'  {plat:20s} {mean_drift:.3f}')

In [ ]:
fig = report.plot_drift()
plt.show()

## 4. Per-cell harmonization confidence

Cells whose raw scores had to shift a lot to align (low confidence) are
candidates for downstream QC triage — often driven by read-depth or
coverage outliers.

In [ ]:
conf = report.confidence
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(conf.to_numpy(), bins=30, color='steelblue', alpha=0.75)
axes[0].set_xlabel('harmonization confidence')
axes[0].set_ylabel('cells')
axes[0].set_title('Per-cell confidence distribution')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(report.alignment.per_cell_shift, conf, s=6, alpha=0.5)
axes[1].set_xlabel('|shift| (pathways averaged)')
axes[1].set_ylabel('harmonization confidence')
axes[1].set_title('Shift vs confidence (monotone by design)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Multi-seed acceptance

The roadmap target (`post rho > 0.75`) is evaluated across many seeds to
absorb Monte Carlo variability. The benchmark's `run_many` is the single
call that produces the summary.

In [ ]:
summary = bench.run_many(n_seeds=5)
summary

## Further reading

- Rosen J et al. *Universal Cell Embeddings: A Foundation Model for
  Cell Biology.* Nature 2024.
- PSF v0.6 roadmap — Phase 1 Rigor Layer F2:
  [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- API guide: [docs/guides/cross-platform.md](../../docs/guides/cross-platform.md)